# GAIA Agent (LangGraph + Gemini)

Hugging Face Agents Course — Unit 4 final assignment.

## Step 0: Install



In [ ]:
%pip install langgraph langchain-core langchain-google-genai python-dotenv requests ddgs pandas openpyxl beautifulsoup4 markdownify

## Step 0: Imports + LLM

In [ ]:
import os
import re
import io
import json
import time
import base64
import contextlib
import mimetypes
from typing import TypedDict, Annotated, Optional

import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from markdownify import markdownify
from ddgs import DDGS

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

load_dotenv()  # reads GOOGLE_API_KEY from .env


In [ ]:
def as_text(message) -> str:
    # Gemini 3 can return .content as a list of parts; .text keeps only the text.
    text = message.text
    return text() if callable(text) else text

### Pick a model that responds

*Each model gets one quick try (no retries, 30 s timeout) and shows how long it took. The first one that answers becomes `MODEL_NAME`.*

In [ ]:
CANDIDATE_MODELS = ["gemini-3.8-flash", "gemini-3.8-flash-lite", "gemini-flash-latest", "gemini-flash-lite-latest"]

MODEL_NAME = None
for name in CANDIDATE_MODELS:
    try:
        start = time.time()
        m = ChatGoogleGenerativeAI(model=name, temperature=0, max_retries=0, timeout=30)
        print("✅", name, "→", as_text(m.invoke("Capital of France? One word.")), f"({time.time() - start:.1f}s)")
        if MODEL_NAME is None:
            MODEL_NAME = name          # use the first model that answers
    except Exception as e:
        print("❌", name, "→", str(e)[:100])

if MODEL_NAME is None:
    MODEL_NAME = CANDIDATE_MODELS[0]
    print("⚠️ No model answered - falling back to", MODEL_NAME)
print("\nUsing:", MODEL_NAME)

In [ ]:
# MODEL_NAME = "gemini-flash-latest"   # uncomment to force a specific model
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,     # most likely answer every time (good for exact match)
    max_retries=3,     # retry on 503 "high demand" / 429 "too many requests"
    timeout=120,       # never hang forever
)

##  code check


In [ ]:
print(as_text(llm.invoke("What is the capital of France? One word.")))

## Step 1: State

*the agent's memory*

In [ ]:
class GaiaState(TypedDict):
    task_id: str
    question: str
    file_path: Optional[str]                              # attached file, or None
    messages: Annotated[list[AnyMessage], add_messages]   # add_messages = APPEND, not replace
    final_answer: Optional[str]                           # cleaned answer to submit

## Step 2: Tools

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (GAIA agent - Hugging Face Agents Course study project)"}

## DuckDuckGo websearch tool

In [ ]:
@tool
def web_search(query: str) -> str:
    """Search the web with DuckDuckGo. Returns the top results as title, URL and a short snippet.
    Use fetch_webpage on a URL to read the full page."""
    try:
        results = DDGS().text(query, max_results=6)
    except Exception as e:
        return f"Search failed: {e}"
    if not results:
        return "No results found. Try different keywords."
    return "\n\n".join(f"{r['title']}\n{r['href']}\n{r['body']}" for r in results)

## Wikepedia search tool

In [ ]:
@tool
def wikipedia_search(query: str) -> str:
    """Search English Wikipedia. Returns matching article titles, URLs and snippets.
    Use fetch_webpage on a URL to read the full article, including its tables."""
    params = {"action": "query", "list": "search", "srsearch": query, "format": "json", "srlimit": 5}
    try:
        data = requests.get("https://en.wikipedia.org/w/api.php", params=params,
                            headers=HEADERS, timeout=30).json()
    except Exception as e:
        return f"Wikipedia search failed: {e}"
    hits = data.get("query", {}).get("search", [])
    if not hits:
        return "No Wikipedia articles found. Try different keywords."
    lines = []
    for hit in hits:
        title = hit["title"]
        snippet = BeautifulSoup(hit.get("snippet", ""), "html.parser").get_text()
        url = "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")
        lines.append(f"{title}\n{url}\n{snippet}")
    return "\n\n".join(lines)

## Webpage_fetch tool

In [ ]:
@tool
def fetch_webpage(url: str, start: int = 0) -> str:
    """Read a web page as Markdown text (tables are kept).
    Long pages are returned in chunks of 20000 characters: call again with start=20000, 40000, ... to read further."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except Exception as e:
        return f"Could not fetch {url}: {e}"

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    main = soup.find("main") or soup.body or soup
    text = markdownify(str(main))
    text = re.sub(r"\n{3,}", "\n\n", text)

    chunk = text[start:start + 20000]
    remaining = len(text) - (start + 20000)
    if remaining > 0:
        chunk += f"\n\n[... {remaining} more characters. Call fetch_webpage again with start={start + 20000}]"
    return chunk

## Python tool

In [ ]:
@tool
def run_python(code: str) -> str:
    """Run Python code and return everything it prints. Always use print() to show results.
    Use it for math, counting, dates, string manipulation and for reading attached files (pandas is available)."""
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            exec(code, {"__name__": "__main__"})
    except Exception as e:
        return buffer.getvalue() + f"\nError: {type(e).__name__}: {e}"
    return buffer.getvalue() or "(no output - did you forget print()?)"

## youtueb video description tool

In [ ]:
@tool
def ask_about_youtube_video(url: str, question: str) -> str:
    """Watch a YouTube video and answer a question about what is shown or said in it.
    Ask a precise question, e.g. 'What is the highest number of birds visible at the same time?'"""
    message = HumanMessage(content=[
        {"type": "text", "text": question},
        {"type": "media", "file_uri": url, "mime_type": "video/mp4"},
    ])
    try:
        return as_text(llm.invoke([message]))
    except Exception as e:
        return f"Could not analyse the video: {e}"

In [ ]:
tools = [web_search, wikipedia_search, fetch_webpage, run_python, ask_about_youtube_video]
llm_with_tools = llm.bind_tools(tools)

## Step 3: Nodes

In [ ]:
SYSTEM_PROMPT = """You are a general AI assistant answering questions from the GAIA benchmark.
Work step by step. Use your tools to look things up and to calculate - never guess a fact you can check.
If a tool result is not enough, search again with different words or open another page.

When you are done, end your reply with one line in exactly this form:
FINAL ANSWER: [YOUR FINAL ANSWER]

YOUR FINAL ANSWER must be a number OR as few words as possible OR a comma separated list of numbers and/or strings.
- Number: no thousands separators, no units ($, %, km...) unless the question asks for them.
- String: no articles, no abbreviations (write city names in full), write digits in plain text unless told otherwise.
- List: apply the rules above to every element and separate elements with ", ".
Follow every formatting instruction in the question exactly (ordering, rounding, plural, capitalisation...)."""

TEXT_EXTENSIONS = {".txt", ".py", ".csv", ".json", ".md", ".html", ".xml", ".tsv"}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".webp"}
AUDIO_EXTENSIONS = {".mp3", ".wav", ".m4a", ".flac", ".ogg"}

In [ ]:
def read_question(state: GaiaState):
    """Prepare the question (and its attached file) for the LLM."""
    question = state["question"]
    file_path = state.get("file_path")
    print(f"📩 Task {state['task_id']}: {question[:100]}")

    text = question
    extra_parts = []

    if file_path:
        ext = os.path.splitext(file_path)[1].lower()
        text += f"\n\nAn attached file is saved at: {os.path.abspath(file_path)}"

        if ext in (".xlsx", ".xls"):
            sheets = pd.read_excel(file_path, sheet_name=None)
            table = "\n\n".join(f"Sheet '{name}':\n{df.to_string()}" for name, df in sheets.items())
            text += f"\nIts contents:\n{table[:30000]}"

        elif ext in TEXT_EXTENSIONS:
            with open(file_path, encoding="utf-8", errors="replace") as f:
                text += f"\nIts contents:\n{f.read()[:30000]}"

        elif ext in IMAGE_EXTENSIONS or ext in AUDIO_EXTENSIONS:
            kind = "image" if ext in IMAGE_EXTENSIONS else "audio"
            mime = mimetypes.guess_type(file_path)[0] or ("image/png" if kind == "image" else "audio/mpeg")
            with open(file_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode("utf-8")
            text += f"\nThe {kind} file is attached to this message."
            extra_parts.append({"type": kind, "base64": b64, "mime_type": mime})

        else:
            text += "\n(Use run_python to open it.)"
        print(f"   📎 attached file: {os.path.basename(file_path)}")

    human = HumanMessage(content=[{"type": "text", "text": text}] + extra_parts)
    return {"messages": [SystemMessage(content=SYSTEM_PROMPT), human]}

In [ ]:
def assistant(state: GaiaState):
    """The LLM looks at the whole conversation and either calls tools or gives the final answer."""
    response = llm_with_tools.invoke(state["messages"])
    for call in response.tool_calls:
        print(f"   🔧 {call['name']}({str(call['args'])[:90]})")
    return {"messages": [response]}

In [ ]:
def extract_answer(state: GaiaState):
    """Pull the clean answer out of the LLM's last message."""
    text = as_text(state["messages"][-1]).strip()

    matches = re.findall(r"FINAL ANSWER:\s*(.+)", text, flags=re.IGNORECASE)
    if matches:
        answer = matches[-1]
    else:
        answer = text.splitlines()[-1] if text else ""

    junk = "*`\"' "
    answer = answer.strip().strip(junk).rstrip(".").strip(junk)
    print(f"   ✅ answer: {answer}")
    return {"final_answer": answer}

## Step 4: Router

In [ ]:
def route_after_assistant(state: GaiaState) -> str:
    """Decide where to go after the LLM has spoken."""
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "use_tool"
    return "done"

## Step 5: Build the graph

In [ ]:
gaia_graph = StateGraph(GaiaState)

gaia_graph.add_node("read_question", read_question)
gaia_graph.add_node("assistant", assistant)
gaia_graph.add_node("tools", ToolNode(tools))
gaia_graph.add_node("extract_answer", extract_answer)

gaia_graph.add_edge(START, "read_question")
gaia_graph.add_edge("read_question", "assistant")
gaia_graph.add_conditional_edges(
    "assistant",
    route_after_assistant,
    {"use_tool": "tools", "done": "extract_answer"},
)
gaia_graph.add_edge("tools", "assistant")      # the loop
gaia_graph.add_edge("extract_answer", END)

compiled_graph = gaia_graph.compile()

try:
    display(Image(compiled_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(compiled_graph.get_graph().draw_mermaid())

## Step 6: Test on one question

In [ ]:
def run_agent(task_id: str, question: str, file_path: Optional[str] = None, verbose: bool = False) -> str:
    result = compiled_graph.invoke(
        {
            "task_id": task_id,
            "question": question,
            "file_path": file_path,
            "messages": [],
            "final_answer": None,
        },
        config={"recursion_limit": 30},
    )
    if verbose:
        for message in result["messages"]:
            message.pretty_print()
    return result["final_answer"]

In [ ]:
answer = run_agent("test-1", "In what year was the Eiffel Tower completed? Multiply that year by 3.", verbose=True)
print("\nSubmitted answer would be:", repr(answer))

## Step 7: Get the real questions

In [ ]:
API_URL = "https://agents-course-unit4-scoring.hf.space"

questions = requests.get(f"{API_URL}/questions", timeout=30).json()
print(f"{len(questions)} questions\n")
for i, q in enumerate(questions, 1):
    print(f"{i:2}. [{q.get('file_name') or '-'}] {q['question'][:90]}")

In [ ]:
def download_file(q: dict) -> Optional[str]:
    """Download the question's attached file (if any) and return its local path."""
    if not q.get("file_name"):
        return None
    os.makedirs("files", exist_ok=True)
    path = os.path.join("files", q["file_name"])
    if not os.path.exists(path):
        response = requests.get(f"{API_URL}/files/{q['task_id']}", timeout=60)
        response.raise_for_status()
        with open(path, "wb") as f:
            f.write(response.content)
    return path

### Try ONE real question first (change 0 → 1, 2, ...)

In [ ]:
q = questions[0]
print(run_agent(q["task_id"], q["question"], download_file(q), verbose=True))

## Step 8: Run all questions

*saved after each one; re-run to continue*

In [ ]:
ANSWERS_FILE = "answers.json"

if os.path.exists(ANSWERS_FILE):
    with open(ANSWERS_FILE) as f:
        answers = json.load(f)
else:
    answers = {}

for i, q in enumerate(questions, 1):
    task_id = q["task_id"]
    if answers.get(task_id):
        continue

    print(f"\n[{i}/{len(questions)}]")
    try:
        answers[task_id] = run_agent(task_id, q["question"], download_file(q))
    except Exception as e:
        print(f"   ❌ {type(e).__name__}: {e}")
        answers[task_id] = ""

    with open(ANSWERS_FILE, "w") as f:
        json.dump(answers, f, indent=2)
    time.sleep(5)

print(f"\nAnswered {sum(1 for a in answers.values() if a)} of {len(questions)}")

## Step 9: Submit



In [ ]:
USERNAME = "SARANMANOJ"
AGENT_CODE = "https://huggingface.co/SARANMANOJ/gaia-agent/tree/main"   # change to your repo

payload = {
    "username": USERNAME,
    "agent_code": AGENT_CODE,
    "answers": [{"task_id": tid, "submitted_answer": ans} for tid, ans in answers.items()],
}

response = requests.post(f"{API_URL}/submit", json=payload, timeout=120)
print(json.dumps(response.json(), indent=2))